# Crypto Lab 06B — RSA

**CYBR 3570: Applied Cryptography**  
**Topic:** RSA, trapdoor permutations, OAEP, PSS, and implementation risks

## Today's big question

> Why is “uses RSA” not enough information to know whether a system is secure?

RSA is one of the most famous public-key cryptosystems, but the core mathematical operation is only the beginning. This notebook separates toy RSA math from secure library-based RSA usage.

## Learning objectives

By the end of this notebook, you should be able to:

1. Explain the difference between symmetric-key and public-key encryption.
2. Generate a toy RSA key pair from small primes.
3. Compute Euler's totient `φ(n)` and the private exponent `d`.
4. Demonstrate textbook RSA encryption and decryption on small integers.
5. Explain why textbook RSA is deterministic and malleable.
6. Use RSA-OAEP with a professional cryptographic library.
7. Use RSA-PSS to sign and verify messages.
8. Explain why RSA should not be implemented from scratch in production systems.

## Setup

This notebook uses only standard Python for the toy examples. The production-style examples use the `cryptography` package.

In [1]:
import math
import time
from dataclasses import dataclass
from typing import Tuple, Optional

print("Lab 06B setup complete")

Lab 06B setup complete


## Part 1 — RSA as a trapdoor permutation

RSA operates on integers modulo `n`, where `n = p * q` for two large secret primes.

The public operation is:

$$y = x^e \mod n$$

The private trapdoor operation is:

$$x = y^d \mod n$$

The values `n` and `e` are public. The value `d` is private. The private exponent works because:

$$ed \equiv 1 \pmod{
arphi(n)}$$

where, for RSA moduli `n = pq`,

$$
arphi(n) = (p-1)(q-1)$$

### Exercise 1 — Concept check

In 2–4 sentences, explain why RSA is called a public-key system. Your answer should distinguish the public key from the private key.

Answer: Because the key used to encrypt is known to the general public. The private key is only known to the key-holder.

## Part 2 — Building a toy RSA key

The following functions implement the basic math needed for toy RSA.

These examples are intentionally small and insecure. They exist only to make the algebra visible.

In [2]:
def egcd(a: int, b: int) -> Tuple[int, int, int]:
    """Extended Euclidean algorithm.

    Returns (g, x, y) such that ax + by = g = gcd(a, b).
    """
    if b == 0:
        return (a, 1, 0)
    g, x1, y1 = egcd(b, a % b)
    return (g, y1, x1 - (a // b) * y1)


def mod_inverse(a: int, m: int) -> int:
    """Return the modular inverse of a modulo m."""
    g, x, _ = egcd(a, m)
    if g != 1:
        raise ValueError(f"{a} has no inverse modulo {m}")
    return x % m


def phi_for_two_primes(p: int, q: int) -> int:
    """Euler's totient for n = p*q where p and q are prime."""
    return (p - 1) * (q - 1)

print("Inverse of 17 mod 3120:", mod_inverse(17, 3120))

Inverse of 17 mod 3120: 2753


### Exercise 2 — Compute toy RSA parameters

Use `p = 61`, `q = 53`, and `e = 17`.

Compute:

- `n = p * q`
- `phi = (p - 1) * (q - 1)`
- `d = e^-1 mod phi`

Then verify that `(e * d) % phi == 1`.

In [4]:
# TODO: Complete this toy RSA setup.
p = 61
q = 53
e = 17

n = p * q
phi = (p-1) * (q-1)
d = mod_inverse(e, phi)

print("n:", n)
print("phi:", phi)
print("d:", d)
print("check:", (e*d) % phi == 1)

n: 3233
phi: 3120
d: 2753
check: True


## Part 3 — Textbook RSA encryption and decryption

Textbook RSA directly applies the trapdoor permutation to a message integer.

Encryption:

$$c = m^e \mod n$$

Decryption:

$$m = c^d \mod n$$

Again: this is educational only. Textbook RSA is not secure encryption.

In [5]:
def rsa_encrypt_textbook(m: int, e: int, n: int) -> int:
    """Toy textbook RSA encryption."""
    if not 0 <= m < n:
        raise ValueError("message must be in range 0 <= m < n")
    return pow(m, e, n)


def rsa_decrypt_textbook(c: int, d: int, n: int) -> int:
    """Toy textbook RSA decryption."""
    return pow(c, d, n)

# Worked example using the parameters from the previous exercise.
p_ex, q_ex, e_ex = 61, 53, 17
n_ex = p_ex * q_ex
phi_ex = phi_for_two_primes(p_ex, q_ex)
d_ex = mod_inverse(e_ex, phi_ex)

m = 65
c = rsa_encrypt_textbook(m, e_ex, n_ex)
recovered = rsa_decrypt_textbook(c, d_ex, n_ex)

print("message:", m)
print("ciphertext:", c)
print("recovered:", recovered)

message: 65
ciphertext: 2790
recovered: 65


### Exercise 3 — Encrypt and decrypt three messages

Using your toy key from Exercise 2, encrypt and decrypt three integer messages smaller than `n`.

Show that decryption recovers the original message each time.

In [35]:
# TODO: Add three integer messages smaller than n.
messages = [1000, 2131, 3232]

for m in messages:
    c = rsa_encrypt_textbook(m, e, n)
    recovered = rsa_decrypt_textbook(c, d, n)
    print(f"m={m}, c={c}, recovered={recovered}, ok={m == recovered}")

m=1000, c=175, recovered=1000, ok=True
m=2131, c=199, recovered=2131, ok=True
m=3232, c=3232, recovered=3232, ok=True


## Part 4 — Why textbook RSA is deterministic

A secure encryption scheme should not reveal when the same plaintext has been encrypted twice.

Textbook RSA fails this test because it is deterministic: the same message encrypted under the same public key always produces the same ciphertext.

In [36]:
# Worked example: deterministic encryption
m = 42
c1 = rsa_encrypt_textbook(m, e_ex, n_ex)
c2 = rsa_encrypt_textbook(m, e_ex, n_ex)
print("first encryption: ", c1)
print("second encryption:", c2)
print("same ciphertext?", c1 == c2)

first encryption:  2557
second encryption: 2557
same ciphertext? True


### Exercise 4 — Explain determinism

Why is deterministic public-key encryption a problem when the set of possible messages is small? Give an example scenario.

Answer: Same information should be encrypted differently so we cannot see any patterns.

## Part 5 — Textbook RSA is malleable

Textbook RSA has a multiplicative property:

$$Enc(m_1) \cdot Enc(m_2) \equiv Enc(m_1 \cdot m_2) \pmod n$$

That means an attacker can combine ciphertexts to create a new ciphertext related to the original plaintexts, even without knowing the private key.

In [37]:
# Worked example: textbook RSA malleability
m1 = 7
m2 = 9

c1 = rsa_encrypt_textbook(m1, e_ex, n_ex)
c2 = rsa_encrypt_textbook(m2, e_ex, n_ex)

combined_ciphertext = (c1 * c2) % n_ex
combined_plaintext = rsa_decrypt_textbook(combined_ciphertext, d_ex, n_ex)

print("m1:", m1)
print("m2:", m2)
print("m1*m2 mod n:", (m1 * m2) % n_ex)
print("decrypted combined ciphertext:", combined_plaintext)

m1: 7
m2: 9
m1*m2 mod n: 63
decrypted combined ciphertext: 63


### Exercise 5 — Demonstrate malleability yourself

Choose two new messages `m1` and `m2`. Encrypt them separately, multiply the ciphertexts modulo `n`, and decrypt the result.

Confirm that the decrypted value equals `(m1 * m2) % n`.

In [39]:
# TODO: Pick two messages and demonstrate textbook RSA malleability.
m1 = 18
m2 = 52

c1 = rsa_encrypt_textbook(m1, e_ex, n_ex)
c2 = rsa_encrypt_textbook(m2, e_ex, n_ex)

combined_text = c1*c2 % n_ex

combined_plain = rsa_decrypt_textbook( combined_text, d_ex, n_ex)

print (combined_plain)
print ("m1 = ", combined_plain / m2)


936
m1 =  18.0


## Part 6 — Fast modular exponentiation

RSA depends on modular exponentiation. A naive implementation would multiply repeatedly, which is inefficient for large exponents.

Python's built-in `pow(base, exponent, modulus)` uses efficient modular exponentiation.

In [41]:
def exp_mod_naive(x: int, e: int, n: int) -> int:
    """Very slow modular exponentiation for demonstration."""
    if e == 0:
        return 1 % n
    y = 1
    for _ in range(e):
        y = (y * x) % n
    return y


def exp_mod_square_and_multiply(x: int, e: int, n: int) -> int:
    """Square-and-multiply modular exponentiation."""
    result = 1
    base = x % n
    while e > 0:
        if e & 1:
            result = (result * base) % n
        base = (base * base) % n
        e >>= 1
    return result

x, exponent, modulus = 3, 65537, 36567232109354321
print(exp_mod_naive(x, exponent, modulus))
print(pow(x, exponent, modulus))
print(exp_mod_square_and_multiply(x, exponent, modulus))

26652909283612267
26652909283612267
26652909283612267


### Exercise 6 — Compare exponentiation strategies

Time the naive function, the square-and-multiply function, and Python's built-in `pow()` for a moderate exponent such as `10_000` or `50_000`.

Do not choose an exponent so large that your notebook becomes slow.

In [43]:
# TODO: Compare runtimes.
base = 7
exponent = 10_000_000
modulus = 999_983

# Your timing code here
start = time.perf_counter()
print(exp_mod_naive(x, exponent, modulus))
elapsed = time.perf_counter() - start
print(f"  time: {elapsed:.6f} seconds")

start = time.perf_counter()
print(pow(x, exponent, modulus))
elapsed = time.perf_counter() - start
print(f"  time: {elapsed:.6f} seconds")

start = time.perf_counter()
print(exp_mod_square_and_multiply(x, exponent, modulus))
elapsed = time.perf_counter() - start
print(f"  time: {elapsed:.6f} seconds")

883449
  time: 0.228113 seconds
883449
  time: 0.000455 seconds
883449
  time: 0.000029 seconds


## Part 7 — RSA-OAEP using a library

Modern RSA encryption should use a padding scheme such as OAEP.

In real systems, RSA is often used to encrypt a symmetric key, while the actual data is encrypted with a symmetric authenticated cipher such as AES-GCM or ChaCha20-Poly1305.

In [44]:
try:
    from cryptography.hazmat.primitives.asymmetric import rsa, padding
    from cryptography.hazmat.primitives import hashes, serialization
    CRYPTOGRAPHY_AVAILABLE = True
except Exception as exc:
    CRYPTOGRAPHY_AVAILABLE = False
    print("cryptography package is not available:", exc)

CRYPTOGRAPHY_AVAILABLE

True

In [61]:
if CRYPTOGRAPHY_AVAILABLE:
    private_key = rsa.generate_private_key(
        public_exponent=65537,
        key_size=2048,
    )
    public_key = private_key.public_key()

    symmetric_key = b"example AES key!!"  # 16 bytes; demonstration only

    ciphertext = public_key.encrypt(
        symmetric_key,
        padding.OAEP(
            mgf=padding.MGF1(algorithm=hashes.SHA256()),
            algorithm=hashes.SHA256(),
            label=None,
        ),
    )

    recovered_key = private_key.decrypt(
        ciphertext,
        padding.OAEP(
            mgf=padding.MGF1(algorithm=hashes.SHA256()),
            algorithm=hashes.SHA256(),
            label=None,
        ),
    )

    print(ciphertext)
    print("ciphertext length:", len(ciphertext))
    print("recovered key:", recovered_key)
    print("matches?", recovered_key == symmetric_key)

b'\x8c87\xe4t7\xf2\xd6\xb8\x83\x0c"67E\xcert\x8a\rc}\x9aj\xa4\xf4\xe5\x1e\xca\xf9\x97\x86\xe7!\xe5\xed\x91\x16\xbf\xdb\x81\x91\xc2\xc7\x961\x11\x07\x89-\xa0P\xc6\xcc\x87l\xd2\xdc\xaedQ\xff\xf9%\xc79\xd3\x88v\x9cr\x97\xc0c\xa4\xcb\xfa\xba\xb6\xef,9\x89Y\xf2\x9c\x8aC\xb9\x01f\x0b\x05\xb5fe\xbcE\x05\x84+o\x1a\xbaO\xbb;\x80\xd9\xde0\xf3h#R=\x7f\xca\x80=\xa9\x11\r\xff;7\xc8\xdf\xa5wX\xd0O\xb4\x89\xb5\xaa\xe1\x08\x05\x90\xa9^\x05\xa6\xcc\xa8y\x03.-\x00\xca\xf44\x81\xc9\x81\x888\xdb\xf5YK\xbb\x8a\x8d418\x90\xae7//\xb7J\xd8\xf5\xeaO\xee%\xad\xf9$$~\xa5\x16rd\x17\x89$_\xdaX\x87k\xef\x8b@m\xc3s\x86B\x11\x81\xdd\xa3\x1d\xed\xae\xc3\xab4*\x1e\r3\x0fP\xfe\xb7\xba\x8e\xc8\xae\x7f7\xf0z\nxd\\8\x0cX\xd0\xab\xfdym\xf5\xd4^\x8bB\xa4\xb2#Ln'
ciphertext length: 256
recovered key: b'example AES key!!'
matches? True


### Exercise 7 — OAEP behavior

Run the previous encryption cell more than once. Does the ciphertext change when encrypting the same `symmetric_key` under the same public key?

Explain why that behavior is desirable.

Answer: That change is desirable because it means the encryption is randomized, not a deterministic function of the public key and the symmetric key. OAEP mixes the message with fresh random padding before the RSA exponentiation. The same symmetric key under the same public key therefore yields a different ciphertext each time, and an attacker who does not know the private key cannot tell whether two ciphertexts hide the same key.

## Part 8 — RSA-PSS signatures using a library

RSA signatures should use a signature scheme such as PSS. A signature is not the same as encryption.

Encryption aims to protect confidentiality. A signature aims to provide authenticity and help prevent forgery.

In [62]:
if CRYPTOGRAPHY_AVAILABLE:
    message = b"Approve transfer of $100.00"

    signature = private_key.sign(
        message,
        padding.PSS(
            mgf=padding.MGF1(hashes.SHA256()),
            salt_length=padding.PSS.MAX_LENGTH,
        ),
        hashes.SHA256(),
    )

    print("signature length:", len(signature))

    # Verification succeeds silently if the signature is valid.
    try:
        public_key.verify(
            signature,
            message,
            padding.PSS(
                mgf=padding.MGF1(hashes.SHA256()),
                salt_length=padding.PSS.MAX_LENGTH,
            ),
            hashes.SHA256(),
        )
        print("signature verified")
    except:
        print("signature not valid")

signature length: 256
signature verified


### Exercise 8 — Signature failure

Modify the signed message by one byte and verify that the original signature no longer verifies.

Catch the exception and print a clear message.

In [54]:
# TODO: Demonstrate that changing the message invalidates the signature.
if CRYPTOGRAPHY_AVAILABLE:
    tampered_message = b"Approve transfer of $900.00"
    # Your verification code here
    try:
        public_key.verify(
            signature,
            tampered_message,
            padding.PSS(
                mgf=padding.MGF1(hashes.SHA256()),
                salt_length=padding.PSS.MAX_LENGTH,
            ),
            hashes.SHA256(),
        )
        print("signature verified")
    except:
        print("signature not valid")

signature not valid


## Part 9 — Engineering checkpoint

Answer the following in your own words.

1. Why should textbook RSA not be used for encryption?
- It is deterministic, so the same message always encrypts to the same ciphertext, and it has no padding. An attacker can see when messages repeat, encrypt guesses under the public key and compare, and exploit malleability.
2. Why is OAEP used with RSA encryption?
- OAEP mixes the message with fresh random padding, then applies RSA. The same plaintext under the same public key produces a different ciphertext each time, and the construction has a security reduction to the RSA problem.
3. Why is PSS used with RSA signatures?
- PSS hashes the message with a random salt and a padding structure before the private-key operation. Verification checks that structure.
4. Why is e = 65537 common?
- It is a Fermat prime so exponentiation needs 16 squarings and one multiply. It's fast to do, and it is large enough to avoid the attacks that hit tiny exponents when padding is missing or weak.
5. What should a security review check besides the word RSA?
- Modulus size, P & Q are large/random/similar in size, appropriate padding, key generation from CSPRNG, private key not in source or logs, and public exponent and modulus actually matching the private key.

## Submission checklist

Before submitting, make sure you have:

- [ ] Completed all written responses.
- [ ] Completed all TODO code cells.
- [ ] Run the notebook from top to bottom without errors.
- [ ] Updated your engineering log.

## Final reflection

Return to today's big question:

> Why is “uses RSA” not enough information to know whether a system is secure?

Answer using at least three details from this notebook.

Relfection: “Uses RSA” only names the trapdoor permutation m^e mod N. Security depends on everything around that operation. A 512-bit modulus, textbook encryption with no padding, e = 3 and a short message, a private key committed to source control, or RSA used to encrypt bulk data directly can all be described as “RSA” and all be broken. A 2048-bit modulus, primes from a CSPRNG, OAEP for key wrapping, PSS for signatures, and AES-GCM for the payload is also “RSA,” and it is a reasonable design. The algorithm name does not fix the modulus size, the padding, the randomness of p and q, whether the private key stays secret, or whether the protocol will accept a downgraded key.